# 10 Fairness Analysis

This notebook summarizes protected-attribute fairness diagnostics for SEX under the final XGBoost public model.

## Reproducibility note

These notebooks are lightweight narrative companions to the source-controlled pipeline. The canonical workflow lives in src, generated metrics and figures live under reports, and the final selected model is models/xgboost_public.pkl. The protected attribute SEX is excluded from active model training and retained only for fairness diagnostics.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, Markdown

from src.utils import ROOT_DIR, REPORTS_DIR, MODELS_DIR, load_dataset_auto, load_model
from src.data_preprocessing import TARGET_COL

ROOT = ROOT_DIR
REPORTS = REPORTS_DIR
MODELS = MODELS_DIR
PROCESSED_DATA = ROOT / 'data' / 'processed' / 'uci_taiwan_credit_default_processed.csv'

pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')


def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def show_image_if_exists(path: Path, width: int = 900):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing optional image: {path.relative_to(ROOT) if path.is_absolute() else path}')


def load_project_dataset():
    try:
        frame, source = load_dataset_auto()
        return frame, source
    except Exception as exc:
        if PROCESSED_DATA.exists():
            print(f'UCI loader failed; using processed local fallback. Reason: {exc}')
            return pd.read_csv(PROCESSED_DATA), PROCESSED_DATA
        raise

raw_df, DATA_SOURCE = load_project_dataset()
print('Project root:', ROOT)
print('Data source:', DATA_SOURCE)
print('Rows:', f'{len(raw_df):,}')
print('Columns:', raw_df.shape[1])

fairness_csv = pd.read_csv(REPORTS / 'fairness_reports' / 'application_model' / 'xgboost_public_fairness_metrics.csv')
fairness_json = load_json(REPORTS / 'fairness_reports' / 'application_model' / 'xgboost_public_fairness_metrics.json')
deep_dive = load_json(REPORTS / 'fairness_reports' / 'application_model' / 'fairness_deep_dive_summary.json')
display(fairness_csv)
display(pd.DataFrame(deep_dive['protected_attribute_mapping']))
print('Protected attribute:', fairness_json['protected_attribute'])

Project root: D:\PGDBA\Projects\Credit Default Risk\credit-default-xai
Data source: uci:\default_credit_card
Rows: 30,000
Columns: 38


,demographic_parity_difference,equalized_odds_difference,equal_opportunity_difference,disparate_impact_ratio
0,0.022032,0.022508,0.006295,0.975355


,sex_code,sex_group,group
0,1,Male,Male (SEX=1)
1,2,Female,Female (SEX=2)


Protected attribute: SEX


In [2]:
outcomes = pd.read_csv(REPORTS / 'fairness_reports' / 'application_model' / 'group_outcome_analysis_by_sex.csv')
errors = pd.read_csv(REPORTS / 'fairness_reports' / 'application_model' / 'group_error_analysis_by_sex.csv')
display(outcomes[['policy', 'threshold', 'sex_code', 'sex_group', 'predicted_high_risk_rate', 'actual_default_rate']])
display(errors[['policy', 'threshold', 'sex_code', 'sex_group', 'false_positive_rate', 'false_negative_rate']])

,policy,threshold,sex_code,sex_group,predicted_high_risk_rate,actual_default_rate
0,xgboost_baseline_threshold_050,0.50,1,Male,0.128031,0.246278
1,xgboost_baseline_threshold_050,0.50,2,Female,0.105998,0.204875
2,xgboost_recall_threshold_025,0.25,1,Male,0.310932,0.246278
3,xgboost_recall_threshold_025,0.25,2,Female,0.241852,0.204875
4,logistic_baseline_threshold_050,0.50,1,Male,0.336027,0.246278
5,logistic_baseline_threshold_050,0.50,2,Female,0.285127,0.204875
6,dnn_baseline_threshold_050,0.50,1,Male,0.139940,0.246278
7,dnn_baseline_threshold_050,0.50,2,Female,0.106820,0.204875
8,dnn_recall_threshold_030,0.30,1,Male,0.262442,0.246278
9,dnn_recall_threshold_030,0.30,2,Female,0.208710,0.204875


,policy,threshold,sex_code,sex_group,false_positive_rate,false_negative_rate
0,xgboost_baseline_threshold_050,0.50,1,Male,0.054176,0.645941
1,xgboost_baseline_threshold_050,0.50,2,Female,0.047882,0.668449
2,xgboost_recall_threshold_025,0.25,1,Male,0.209368,0.378238
3,xgboost_recall_threshold_025,0.25,2,Female,0.162590,0.450535
4,logistic_baseline_threshold_050,0.50,1,Male,0.237585,0.362694
5,logistic_baseline_threshold_050,0.50,2,Female,0.209783,0.422460
6,dnn_baseline_threshold_050,0.50,1,Male,0.063205,0.625216
7,dnn_baseline_threshold_050,0.50,2,Female,0.049948,0.672460
8,dnn_recall_threshold_030,0.30,1,Male,0.162528,0.431779
9,dnn_recall_threshold_030,0.30,2,Female,0.131244,0.490642


## Interpretation

The fairness deep dive is a governance diagnostic, not proof of legal discrimination or causal bias. Male applicants had higher high-risk flag rates, while the error-rate pattern differs by false-positive and false-negative harm.